---
# 02 - Classify all particles according to the taxonomy

In [1]:
import ast
import json
import os
import random
import re
import subprocess
from pathlib import Path

import dspy
import pandas as pd

In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT    = Path("../../").resolve()
SUMMARY_CSV  = REPO_ROOT / "main_results/allen/modelsmc/sonnet/summary.csv"
RESULTS_ROOT = REPO_ROOT / "results/allen_modelsmc_sonnet"

TAXONOMY_JSON  = Path("allen_taxonomy.json")
CLASSIFIED_CSV = Path("allen_classified.csv")

# ── Tunable constants ──────────────────────────────────────────────────────────
CLASSIF_BATCH_SIZE = 5   # simulators per LLM call
CHECKPOINT_EVERY   = 25  # save CSV after this many newly classified particles
VALIDATION_N       = 20  # particles to spot-check in Phase 5

if not TAXONOMY_JSON.exists():
    raise FileNotFoundError(
        f"{TAXONOMY_JSON} not found. Run 01_extract_taxonomy.ipynb first."
    )

# ── Build simulator index: (seed, iteration, particle_index) -> Path ──────────
# Scans ALL timestamp subdirectories; later timestamp wins on duplicates.
sim_index = {}
timestamp_dirs = sorted(
    [d for d in RESULTS_ROOT.iterdir() if d.is_dir() and not d.name.startswith(".")]
)
for ts_dir in timestamp_dirs:
    for seed_dir in ts_dir.iterdir():
        if not seed_dir.is_dir():
            continue
        try:
            seed_val = int(seed_dir.name)
        except ValueError:
            continue
        for particle_dir in seed_dir.iterdir():
            if not particle_dir.is_dir():
                continue
            name = particle_dir.name
            m = __import__("re").match(r"iter-(\d+)_p(\d+)", name)
            if not m:
                continue
            key = (seed_val, int(m.group(1)), int(m.group(2)))
            sim_index[key] = particle_dir / "simulator.py"
print(f"sim_index built: {len(sim_index)} entries from {len(timestamp_dirs)} timestamp dir(s)")

In [3]:
# ── API key via dotenvx ───────────────────────────────────────────────────────
result = subprocess.run(
    ["dotenvx", "get", "ANTHROPIC_API_KEY"],
    capture_output=True, text=True, cwd=REPO_ROOT,
)
api_key = result.stdout.strip()
if not api_key:
    raise RuntimeError("Could not retrieve ANTHROPIC_API_KEY via dotenvx")
os.environ["ANTHROPIC_API_KEY"] = api_key
print("API key loaded:", api_key[:8] + "...")

lm = dspy.LM(
    model="anthropic/claude-sonnet-4-6",
    api_key=api_key,
    temperature=0.0,
    max_tokens=20_000,
)
dspy.configure(lm=lm)
print("DSPy LM configured:", lm.model)

# ── Load and validate taxonomy ────────────────────────────────────────────────
with open(TAXONOMY_JSON) as f:
    taxonomy = json.load(f)

valid_subtype_ids = {ct["subtype_id"] for ct in taxonomy["taxonomy"]}
valid_subtype_ids.add(taxonomy.get("base_only_subtype_id", "base_only"))
valid_subtype_ids.add(taxonomy.get("unknown_subtype_id",   "unknown"))
taxonomy_str = json.dumps(taxonomy, indent=2)

print(f"\nTaxonomy: {len(taxonomy['taxonomy'])} subtypes")
print("Valid subtype IDs:", sorted(valid_subtype_ids))

API key loaded: sk-ant-a...
DSPy LM configured: anthropic/claude-sonnet-4-6

Taxonomy: 11 subtypes
Valid subtype IDs: ['I_M_fixed_tau', 'I_M_plus_I_A', 'I_M_plus_I_Ks', 'I_M_plus_I_NaP_fixed_tau', 'I_M_plus_I_NaP_voltage_dep_tau', 'I_M_plus_I_bgK', 'I_M_plus_I_h_fixed_tau', 'I_M_plus_I_h_voltage_dep_tau', 'I_M_voltage_dep_tau', 'base_only', 'unknown']


In [4]:
# ── Load summary.csv and build usable particle set ────────────────────────────
df = pd.read_csv(SUMMARY_CSV)

def get_seed(cfg_str):
    return ast.literal_eval(cfg_str)["seed"]

df["seed"] = df["config"].apply(get_seed)

def sim_path(row):
    key = (int(row["seed"]), int(row["iteration"]), int(row["particle_index"]))
    return sim_index.get(key)  # None if not found

df["sim_path"]   = df.apply(sim_path, axis=1)
df["sim_exists"] = df["sim_path"].apply(lambda p: p is not None and p.exists())

df_usable = (
    df[(df["log_weight"] != float("-inf")) & df["sim_exists"]]
    .copy()
    .reset_index(drop=True)
)

print(f"Usable particles: {len(df_usable)}")
print(f"log_weight range: [{df_usable['log_weight'].min():.1f}, {df_usable['log_weight'].max():.1f}]")


def parse_json_response(raw: str, fallback=None):
    """Robust JSON parser: strips markdown fences, unwraps dict wrappers."""
    text = raw.strip()
    if text.startswith("```"):
        text = "\n".join(text.split("\n")[1:])
    if text.endswith("```"):
        text = "\n".join(text.split("\n")[:-1])
    text = text.strip()
    parsed = None
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        m = re.search(r"(\[.*\]|\{.*\})", text, re.DOTALL)
        if m:
            try:
                parsed = json.loads(m.group())
            except json.JSONDecodeError:
                pass
    if parsed is None:
        if fallback is not None:
            print(f"  WARNING: JSON parse failed. Raw: {text[:200]}")
            return fallback
        raise ValueError(f"Could not parse JSON: {text[:500]}")
    if isinstance(parsed, dict):
        for key in ("results", "descriptions", "classifications", "items", "data"):
            if key in parsed and isinstance(parsed[key], list):
                return parsed[key]
    return parsed

Usable particles: 1445
log_weight range: [-2239613952.0, -232.6]


---
## Phase 4 — Full Classification

Classifies every usable particle against the taxonomy in batches of 5.
Progress is checkpointed every 25 particles so you can interrupt and resume freely.

Expected runtime: ~30–60 minutes for 1288 particles.

In [5]:
class ClassifyBatch(dspy.Signature):
    """You are an expert computational neuroscientist.

    The BASE Hodgkin-Huxley model has EXACTLY THREE ionic currents:
      1. Fast Na+ current: m**3 * h gating, E_Na = +53 mV
      2. Delayed-rectifier K+ current: n**4 gating, E_K = -107 mV
      3. Ohmic Leak current: g_leak * (V - E_leak)
    Anything beyond these three is an "additional channel".

    You are given a taxonomy of additional channel types and a batch of simulators.
    For EACH snippet, assign EXACTLY ONE subtype_id from the taxonomy.

    The taxonomy subtypes are defined by structural code properties:
    - activation_power: the exponent of the activation gate (p^1, p^2, p^3, instantaneous, ...)
    - has_inactivation_gate: whether a separate inactivation variable (q, b, h) is present
    - tau_type: how the time constant is computed
        voltage_dependent — tau varies with V (bell-shaped or Boltzmann-derived)
        fixed_scalar      — tau is a constant scalar from a learned parameter
        log_scaled        — tau uses a log/exp mapping from a parameter
        instantaneous     — no dynamic state variable; activation is algebraic
    The taxonomy JSON includes a split_criterion field per subtype to guide you.

    Classification rules:
    - Read the FULL code to determine the gating structure, ion type, and tau implementation.
    - Base your decision on structural differences, not parameter values.
    - Adds nothing beyond base HH  -> use the base_only subtype_id.
    - Adds TWO additional channels -> use the appropriate multi_* subtype_id.
    - Use unknown only if the mechanism is genuinely unrecognisable.

    Return a JSON array, one entry per snippet IN THE SAME ORDER as input:
    [
      {
        "snippet_num": 1,
        "family_id": "I_M",
        "subtype_id": "I_M_voltage_dep_tau",
        "reasoning": "Single gate p, tau_p is bell-shaped function of V -> voltage_dependent tau -> I_M_voltage_dep_tau."
      },
      ...
    ]
    """
    taxonomy_json:        str = dspy.InputField(desc="JSON taxonomy with family_id, subtype_id, split_criterion per entry")
    batch_snippets:       str = dspy.InputField(desc="Full simulator code snippets with ### SNIPPET N (particle_id: ...) ### headers")
    classifications_json: str = dspy.OutputField(desc="JSON array, one object per snippet in order")

In [6]:
# ── Load checkpoint ───────────────────────────────────────────────────────────
if CLASSIFIED_CSV.exists():
    df_existing = pd.read_csv(CLASSIFIED_CSV)
    done_ids    = set(df_existing["particle_id"])
    results     = df_existing.to_dict("records")
    print(f"Resuming: {len(done_ids)}/{len(df_usable)} already classified")
else:
    done_ids = set()
    results  = []

to_classify = df_usable[~df_usable["particle_id"].isin(done_ids)].reset_index(drop=True)
print(f"Remaining: {len(to_classify)} particles to classify")

classify       = dspy.Predict(ClassifyBatch)
new_since_save = 0
n_batches      = (len(to_classify) + CLASSIF_BATCH_SIZE - 1) // CLASSIF_BATCH_SIZE

for batch_idx in range(n_batches):
    start     = batch_idx * CLASSIF_BATCH_SIZE
    batch     = to_classify.iloc[start : start + CLASSIF_BATCH_SIZE]
    batch_ids = []

    snippets_text = ""
    for i, (_, row) in enumerate(batch.iterrows(), 1):
        code_text     = Path(row["sim_path"]).read_text()  # full code — never trim particle simulators
        pid           = row["particle_id"]
        snippets_text += f"### SNIPPET {i} (particle_id: {pid}) ###\n{code_text}\n\n"
        batch_ids.append(pid)

    try:
        res    = classify(taxonomy_json=taxonomy_str, batch_snippets=snippets_text)
        parsed = parse_json_response(res.classifications_json, fallback=[])
    except Exception as e:
        print(f"  Batch {batch_idx+1}/{n_batches}: ERROR — {e}")
        parsed = []

    for j in range(len(batch)):
        row = batch.iloc[j]
        if j < len(parsed):
            item    = parsed[j]
            subtype = item.get("subtype_id", "unknown")
            family  = item.get("family_id",  "unknown")
            reason  = item.get("reasoning",  "")
            if subtype not in valid_subtype_ids:
                print(f'  WARNING: unrecognised subtype_id "{subtype}" -> mapping to unknown')
                subtype = "unknown"
        else:
            subtype = "unknown"
            family  = "unknown"
            reason  = "ERROR: LLM did not return a result for this snippet"

        results.append({
            "particle_id":    row["particle_id"],
            "run_id":         row["run_id"],
            "seed":           int(row["seed"]),
            "iteration":      int(row["iteration"]),
            "particle_index": int(row["particle_index"]),
            "log_weight":     float(row["log_weight"]),
            "family_id":      family,
            "subtype_id":     subtype,
            "reasoning":      reason,
        })
        new_since_save += 1

    if new_since_save >= CHECKPOINT_EVERY:
        pd.DataFrame(results).to_csv(CLASSIFIED_CSV, index=False)
        pct = 100 * len(results) / len(df_usable)
        print(f"  [{len(results)}/{len(df_usable)} = {pct:.0f}%] checkpoint saved")
        new_since_save = 0

pd.DataFrame(results).to_csv(CLASSIFIED_CSV, index=False)
print(f"\nPhase 4 done. {len(results)} particles classified -> {CLASSIFIED_CSV}")

Remaining: 1445 particles to classify
  [25/1445 = 2%] checkpoint saved
  [50/1445 = 3%] checkpoint saved
  [75/1445 = 5%] checkpoint saved
  [100/1445 = 7%] checkpoint saved
  [125/1445 = 9%] checkpoint saved
  [150/1445 = 10%] checkpoint saved
  [175/1445 = 12%] checkpoint saved
  [200/1445 = 14%] checkpoint saved
  [225/1445 = 16%] checkpoint saved
  [250/1445 = 17%] checkpoint saved
  [275/1445 = 19%] checkpoint saved
  [300/1445 = 21%] checkpoint saved
  [325/1445 = 22%] checkpoint saved
  [350/1445 = 24%] checkpoint saved
  [375/1445 = 26%] checkpoint saved
  [400/1445 = 28%] checkpoint saved
  [425/1445 = 29%] checkpoint saved
  [450/1445 = 31%] checkpoint saved
  [475/1445 = 33%] checkpoint saved
  [500/1445 = 35%] checkpoint saved
  [525/1445 = 36%] checkpoint saved
  [550/1445 = 38%] checkpoint saved
  [575/1445 = 40%] checkpoint saved
  [600/1445 = 42%] checkpoint saved
  [625/1445 = 43%] checkpoint saved
  [650/1445 = 45%] checkpoint saved
  [675/1445 = 47%] checkpoint save

In [7]:
# ── Distribution check ────────────────────────────────────────────────────────
df_out = pd.read_csv(CLASSIFIED_CSV)
print(f"Total classified: {len(df_out)}\n")

print("Subtype distribution:")
print(df_out["subtype_id"].value_counts().to_string())

print("\nFamily distribution:")
print(df_out["family_id"].value_counts().to_string())

n_unk = (df_out["subtype_id"] == "unknown").sum()
print(f"\nunknown: {n_unk} ({100 * n_unk / len(df_out):.1f}%)")

Total classified: 1445

Subtype distribution:
subtype_id
I_M_voltage_dep_tau               560
I_M_fixed_tau                     524
I_M_plus_I_NaP_voltage_dep_tau    150
I_M_plus_I_NaP_fixed_tau           66
I_M_plus_I_h_fixed_tau             50
I_M_plus_I_A                       35
I_M_plus_I_h_voltage_dep_tau       30
unknown                            11
base_only                          10
I_M_plus_I_Ks                       7
I_M_plus_I_bgK                      2

Family distribution:
family_id
I_M               1084
I_M_plus_I_NaP     216
I_M_plus_I_h        80
I_M_plus_I_A        35
unknown             11
base_only           10
I_M_plus_I_Ks        7
I_M_plus_I_bgK       2

unknown: 11 (0.8%)


---
## Phase 5 — Validation Spot-Check

Prints the **full simulator code** alongside the assigned label and reasoning.

Use this to catch systematic misclassifications before using `allen_classified.csv`
in `figures/Fig_5_posterior_mass_analysis/notebooks/plotting.ipynb` and
`figures/Fig_F2_cross_seed_rank_stability/notebooks/plotting.ipynb`.

In [ ]:
random.seed(99)
sample_ids = random.sample(df_out["particle_id"].tolist(), min(VALIDATION_N, len(df_out)))
df_val = (
    df_out[df_out["particle_id"].isin(sample_ids)]
    .merge(df_usable[["particle_id", "sim_path"]], on="particle_id", how="left")
)

SEP = "-" * 70
for _, row in df_val.iterrows():
    code_text = Path(row["sim_path"]).read_text()
    excerpt   = code_text  # full code for accurate spot-check
    print(SEP)
    print(f"particle : {str(row['particle_id'])[:8]}...   log_weight = {row['log_weight']:.1f}")
    print(f"label    : [{row['subtype_id']}]   family = [{row['family_id']}]")
    print(f"reasoning: {row['reasoning']}")
    print(f"\nFull simulator code:\n{excerpt}\n")

print(SEP)
print("\nPhase 5 done. Proceed to figures/Fig_5_posterior_mass_analysis/notebooks/plotting.ipynb "
      "and figures/Fig_F2_cross_seed_rank_stability/notebooks/plotting.ipynb.")